<div dir="rtl" align="right">

# NB22D — گسترش بازیابی در کل پیکره

این فایل را با نام `22d_full_corpus_retrieval_expansion.ipynb` در `notebooks/phase2/` قرار بده و **Restart Kernel → Run All** اجرا کن. نوت‌بوک به History کرنل یا اجرای دوبارهٔ NB22B/NB22C متکی نیست.

**مسئلهٔ اثبات‌شده:** سقف نظری Recall@10 در Hybrid100 روی Development برابر ۸۰٫۸۹٪ است؛ حتی ترتیب‌بندی ایدئال در همین Pool به هدف ۸۵٪ نمی‌رسد. NB22C بهبود کوچک رتبه‌بندی داشت، اما مانع پوشش باقی است.

**تغییر جدید:** به‌جای تنظیم دوبارهٔ همان Pool، جست‌وجوی BM25 واژه‌ای و سه‌نویسه‌ای روی محتوای تمام ۳۴۸۸ قطعه انجام می‌شود. کاندیداهای بیرون Hybrid100 می‌توانند وارد Pool400/800 شوند. این مشتق آزمایشی نسخه‌دار است؛ Dense اصلی بازسازی یا تغییر نمی‌کند. اثربخشی این کانال‌های مکمل باید با نتیجهٔ واقعی سنجیده شود و تضمین بهبود بزرگ نیست.

**۵ اقدام؛ هرکدام سه زیرکار، در پنج بستهٔ وابسته:** اتصال ورودی؛ فریز فرضیه؛ جست‌وجوی کامل با کش؛ انتخاب روی tuning؛ بررسی داخلی و خروجی نهایی. Imports، Bootstrap و مکانیک مشترک جدا هستند.

**API، دانلود و اجرای مدل تازه: صفر.** پردازش CPU برای ساخت دو نمایهٔ واژه/نویسه و جست‌وجوی Development انجام می‌شود. زمان تضمین نیست؛ حد یک‌ساعته و کش مستقل هر سؤال دارد. Run All مجدد کش دقیق را می‌خواند؛ Cache miss تنها جست‌وجوی واژه/نویسه است، نه مدل. شاخص و پاسخ‌های قدیمی دست‌نخورده‌اند.

ورودی‌ها از C056 و گزارش/رتبه‌ها/تشخیص اجرای موفق NB22C با هش دقیق خوانده می‌شوند. فایل‌های بزرگ روی دستگاه کاربر لازم‌اند؛ در صورت نبود، مسیر دقیق و BLOCKED چاپ می‌شود. `RAG_FINANCE_ROOT` فقط در صورت پیدا نشدن مخزن لازم است. پوشهٔ جدید خروجی `reports/phase2/quality_improvement/NB22D.v1/` و کش جدید `artifacts/phase2/quality_improvement/NB22D.lexical_cache_v1/` است.

**پذیرش جدول:** Recall@10≥۸۵٪؛ nDCG@10≥۷۵٪؛ Hit@10 حد مطلق مصوب ندارد و فقط عدم‌افت آن بررسی می‌شود. فقط حد مطلقِ محقق‌شده علامت ✅ دارد؛ ستون سایر موارد خالی است. این علامت در دامنهٔ Development است و پذیرش مستقل Release نیست.

دو انتخاب جدا ثبت می‌شود: بهترین Pool برای بازیابی شاهد، و بهترین رتبه‌بندی نهاییِ قابل مقایسه با NB22C. اولی با سقف پوشش سنجیده می‌شود و جای Recall واقعی Top10 نیست. فقط یک برندهٔ هر شاخه پس از فریز روی validation داخلی بررسی می‌شود؛ اگر رتبه‌بندی افت کند، منتخب NB22C حفظ می‌شود. validation از Development قبلاً دیده‌شده است؛ Test تاریخی برای انتخاب/ارزیابی تازه استفاده نمی‌شود. ردیف‌های Test فقط برای کنترل نگاشت و هویت ورودی خوانده می‌شوند.

این نوت‌بوک موتور پاسخ‌گویی را اجرا نمی‌کند. بعد از احراز پوشش کافی، فایل Pool جدید برای پایلوت محدود Reranker آماده است؛ قبل از دیدن نتیجهٔ پوشش، اجرای سنگین مدل شروع نمی‌شود. وضعیت‌های کیفیت، مسائل باز، تعویق اقدام۴۸ و عدم‌پذیرش Release محفوظ‌اند.

</div>

<div dir="rtl" align="right">

### NB22D-C001 — Imports

فقط کتابخانه‌های استاندارد؛ بدون فایل/مدل/شبکه.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [1]:
# NB22D-C001 — Imports only; standard library, no file/network/model operations.
import collections
import hashlib
import json
import math
import os
import platform
import re
import statistics
import tempfile
import time
from pathlib import Path

import unicodedata


<div dir="rtl" align="right">

### NB22D-C002 — Bootstrap

فقط تنظیمات، مسیر و هویت کد؛ بدون نوشتن یا inference.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [2]:
# NB22D-C002 — Explicit offline experiment, no dispatch or inference fallback.
PINS = {'C056': {'path': 'reports/phase2/release/NB22.C056.retrieval_verification/C056.independent_retrieval_metric_verification.ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22.json', 'file_bytes_sha256': 'ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22', 'bytes': 1180779, 'write_status': 'CREATED', 'reload_validation': 'PASS'}, 'C_final': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A06.final_report.1797d76fe08136a50ffa846da39858f29838a9bc8a3dc3d1cad120789ccc53ea.json', 'file_bytes_sha256': '1797d76fe08136a50ffa846da39858f29838a9bc8a3dc3d1cad120789ccc53ea', 'bytes': 31539, 'reload_validation': 'PASS'}, 'C_rankings': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A05.retained_development_rankings.5bc216aae839bc80cc2f184ca3a3cd3e4ac33657655c3bffb4fac410e96e9190.jsonl', 'file_bytes_sha256': '5bc216aae839bc80cc2f184ca3a3cd3e4ac33657655c3bffb4fac410e96e9190', 'bytes': 19285045, 'reload_validation': 'PASS'}, 'C_diagnosis': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A02.development_diagnosis.2b86a0709a76b3e5673383d6f95d973991890246cb129faf7fe995e8f28dcdf5.jsonl', 'file_bytes_sha256': '2b86a0709a76b3e5673383d6f95d973991890246cb129faf7fe995e8f28dcdf5', 'bytes': 1610871, 'reload_validation': 'PASS'}}
CONFIG = {
 'version': 'NB22D_FULL_CORPUS_LEXICAL_EXPANSION_V1',
 'seed': 'NB22C_20261030_SOURCE_COMPONENT_SPLIT_V1',
 'validation_percent': 20, 'minimum_components_per_partition': 30,
 'thresholds': {'recall_at_10': .85, 'ndcg_at_10': .75},
 'bm25': {'k1': 1.2, 'b': .3}, 'char_max_df_fraction': .30,
 'pool_specs': [{'name':'word400','channels':['word'],'depth':400},
                {'name':'char400','channels':['char'],'depth':400},
                {'name':'dual400','channels':['word','char'],'depth':400},
                {'name':'dual800','channels':['word','char'],'depth':800}],
 'ranking_specs': [{'name':'word_rrf','channels':['word']},
                   {'name':'char_rrf','channels':['char']},
                   {'name':'dual_rrf','channels':['word','char']}],
 'rrf_k': 10, 'incumbent_rank_weight': .75,
 'maximum_local_seconds': 3600,
 'api_calls': 0, 'new_model_pairs': 0, 'downloads': False,
}
_start=Path(os.environ.get('RAG_FINANCE_ROOT',str(Path.cwd()))).expanduser().resolve()
ROOT=next((p for p in [_start,*_start.parents] if (p/'notebooks/phase2').is_dir() and
 (p/'data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md').is_file()),None)
STATE={'actions':{},'artifacts':[],'memory':{},'errors':[],'inputs':{},'api_calls':0,'model_pairs':0,'network_calls':0}
NOTEBOOK_CODE_HASH=None;SAVED_CODE_VALID=False
if ROOT is not None:
 p=ROOT/'notebooks/phase2/22d_full_corpus_retrieval_expansion.ipynb'
 if p.is_file():
  n=json.loads(p.read_bytes()); codes=[''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in n['cells'] if c['cell_type']=='code']
  NOTEBOOK_CODE_HASH=hashlib.sha256(json.dumps(codes,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
  SAVED_CODE_VALID=NOTEBOOK_CODE_HASH==n.get('metadata',{}).get('nb22d',{}).get('code_source_sha256')
print(json.dumps({'stage':'BOOTSTRAP','root_resolved':ROOT is not None,'code_identity_valid':SAVED_CODE_VALID,
 'new_api_calls':0,'new_model_pairs':0,'historical_test_evaluation':False},ensure_ascii=False,indent=2))


{
  "stage": "BOOTSTRAP",
  "root_resolved": true,
  "code_identity_valid": true,
  "new_api_calls": 0,
  "new_model_pairs": 0,
  "historical_test_evaluation": false
}


<div dir="rtl" align="right">

### NB22D-C003 — مکانیک مشترک

کنترل هش و JSON، محاسبهٔ سنجه، ذخیرهٔ Atomic و Reload و تفکیک منبع.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [3]:
# NB22D-C003 — Shared mechanics; no historical notebook is executed.
class InputGap(RuntimeError):
    pass

def require(ok, reason):
    if not ok:
        raise InputGap(reason)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def decode(raw):
    def pairs(items):
        out = {}
        for k, v in items:
            if k in out:
                raise InputGap('DUPLICATE_JSON_KEY:' + k)
            out[k] = v
        return out
    def reject(x):
        raise InputGap('NONFINITE_JSON:' + x)
    return json.loads(raw, object_pairs_hook=pairs, parse_constant=reject)

def path_for(relative):
    require(ROOT is not None, 'PROJECT_ROOT_REQUIRED:run_inside_repository_or_set_RAG_FINANCE_ROOT')
    p = Path(relative)
    require(not p.is_absolute() and '..' not in p.parts, 'PROJECT_RELATIVE_PATH_REQUIRED')
    target = (ROOT / p).resolve()
    require(target.is_relative_to(ROOT), 'PATH_OUTSIDE_PROJECT')
    return target

def binding_hash(binding):
    values = {binding[k] for k in ('file_bytes_sha256', 'sha256') if k in binding}
    require(len(values) == 1, 'UNAMBIGUOUS_BYTE_HASH_REQUIRED')
    h = next(iter(values))
    require(isinstance(h, str) and re.fullmatch('[a-f0-9]{64}', h), 'INVALID_HASH')
    return h

def checked_bytes(binding):
    p = path_for(binding['path'])
    require(p.is_file(), 'REQUIRED_FILE_NOT_AVAILABLE:' + binding['path'])
    before = p.stat()
    raw = p.read_bytes()
    after = p.stat()
    require((before.st_size, before.st_mtime_ns, before.st_ino) ==
            (after.st_size, after.st_mtime_ns, after.st_ino), 'SOURCE_CHANGED_DURING_READ')
    require(digest(raw) == binding_hash(binding), 'INPUT_BYTE_HASH_MISMATCH:' + binding['path'])
    STATE['inputs'][binding['path']] = dict(binding)
    return raw

def read_json(binding):
    return decode(checked_bytes(binding))

def read_rows(binding):
    raw = checked_bytes(binding)
    require(all(line.strip() for line in raw.splitlines()), 'BLANK_JSONL_ROW')
    return [decode(line) for line in raw.splitlines()]

def keyed(rows, field):
    out = {}
    for row in rows:
        require(isinstance(row, dict) and isinstance(row.get(field), str) and
                row[field] not in out, 'MISSING_OR_DUPLICATE_ID:' + field)
        out[row[field]] = row
    return out

def save(stem, value, rows=False):
    folder = path_for('reports/phase2/quality_improvement/NB22D.v1')
    folder.mkdir(parents=True, exist_ok=True)
    raw = b''.join(canonical(r) + b'\n' for r in value) if rows else canonical(value)
    suffix = 'jsonl' if rows else 'json'
    p = folder / (stem + '.' + digest(raw) + '.' + suffix)
    if not p.exists():
        fd, name = tempfile.mkstemp(dir=folder, prefix='.nb22d-')
        try:
            with os.fdopen(fd, 'wb') as f:
                f.write(raw); f.flush(); os.fsync(f.fileno())
            try:
                os.link(name, p)
            except FileExistsError:
                pass
        finally:
            Path(name).unlink(missing_ok=True)
    actual = p.read_bytes()
    require(actual == raw, 'OUTPUT_COLLISION_OR_RELOAD_FAILURE')
    if rows:
        require([decode(x) for x in actual.splitlines()] == value, 'OUTPUT_ROW_RELOAD_FAILURE')
    else:
        require(decode(actual) == value, 'OUTPUT_RELOAD_FAILURE')
    receipt = {'path': p.relative_to(ROOT).as_posix(), 'file_bytes_sha256': digest(raw),
               'bytes': len(raw), 'reload_validation': 'PASS'}
    STATE['artifacts'].append(receipt)
    return receipt

def metrics(order, gold):
    require(gold and len(order) == len(set(order)), 'INVALID_METRIC_POPULATION')
    flags = [int(x in gold) for x in order[:50]]
    out = {}
    for k in (5, 10):
        hits = sum(flags[:k])
        idcg = math.fsum(1 / math.log2(i + 2) for i in range(min(k, len(gold))))
        out.update({f'hit_at_{k}': float(hits > 0), f'precision_at_{k}': hits / k,
                    f'recall_at_{k}': hits / len(gold),
                    f'ndcg_at_{k}': math.fsum(v / math.log2(i + 2) for i, v in enumerate(flags[:k])) / idcg})
    first = next((i + 1 for i, v in enumerate(flags) if v), None)
    out['reciprocal_rank'] = 1 / first if first else 0.0
    return out

def mean_metric(values):
    rows = list(values)
    require(bool(rows), 'EMPTY_METRIC_COHORT')
    return {k: math.fsum(r[k] for r in rows) / len(rows) for k in sorted(rows[0])}

def rank_ids(row, count):
    c = row['candidates']
    require(len(c) == count and [r['rank'] for r in c] == list(range(1, count + 1)), 'RANK_SEQUENCE_OR_DEPTH_CONFLICT')
    ids = [r['chunk_id'] for r in c]
    require(len(ids) == len(set(ids)), 'DUPLICATE_RANKED_CHUNK')
    return ids

def run_action(number, fn, dependencies=()):
    blocked = [d for d in dependencies if STATE['actions'].get(d, {}).get('status') != 'COMPLETE_IN_SCOPE']
    if blocked:
        out = {'status': 'BLOCKED_BY_DEPENDENCY', 'dependencies': blocked}
    else:
        try:
            out = fn()
            out['status'] = 'COMPLETE_IN_SCOPE'
        except InputGap as error:
            out = {'status': 'BLOCKED_INPUT_OR_CONTRACT', 'reason': str(error)}
        except Exception as error:
            # Real implementation errors remain distinguishable from missing input.
            out = {'status': 'FAILED_IMPLEMENTATION', 'error_type': type(error).__name__}
            STATE['errors'].append({'action': number, 'error_type': type(error).__name__})
    STATE['actions'][number] = out
    print(json.dumps({'action': number, **out, 'new_api_calls': 0}, ensure_ascii=False, indent=2))
    return out

def components(dev):
    parent = {q: q for q in dev['ids']}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    seen = {}
    for q in dev['ids']:
        for cid in dev['gold'][q]:
            sid = dev['chunks'][cid].get('source_id')
            require(isinstance(sid, str) and sid.strip(), 'GOLD_SOURCE_PROVENANCE_REQUIRED')
            if sid in seen:
                a, b = find(q), find(seen[sid])
                parent[max(a, b)] = min(a, b)
            else:
                seen[sid] = q
    return {q: find(q) for q in dev['ids']}

def partition(component_map):
    labels = {c: ('validation' if int(digest((CONFIG['seed'] + c).encode())[:16], 16) % 100
                    < CONFIG['validation_percent'] else 'tuning') for c in set(component_map.values())}
    out = {name: sorted(q for q, c in component_map.items() if labels[c] == name)
           for name in ('tuning', 'validation')}
    for name, ids in out.items():
        require(len({component_map[q] for q in ids}) >= CONFIG['minimum_components_per_partition'],
                'INSUFFICIENT_SOURCE_COMPONENTS:' + name)
    return out

def eligible(candidate, reference):
    return all(candidate[k] >= reference[k] - 1e-12 for k in ('recall_at_10', 'hit_at_10')) and candidate['ndcg_at_10'] > reference['ndcg_at_10'] + 1e-12



<div dir="rtl" align="right">

### NB22D-C004 — اقدام۱ — اتصال ورودی

۱٫۱ ورودی دقیق؛ ۱٫۲ Gold و رتبه‌های منتخب؛ ۱٫۳ حفظ تقسیم داخلی NB22C. پایان: تطبیق کامل هش، جمعیت و سنجه.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [4]:
# NB22D-C004 — Action 1: exact corpus, Golden and NB22C retained candidate.
def action01():
 require(SAVED_CODE_VALID,'SAVED_NOTEBOOK_CODE_IDENTITY_REQUIRED')
 verified=read_json(PINS['C056']); prior=read_json(PINS['C_final'])
 require(verified['independent_recomputation_complete'] and verified['all_recorded_metric_comparisons_match'],'C056_VERIFICATION_REQUIRED')
 require(prior['release_acceptance_granted'] is False,'PRIOR_ACCEPTANCE_SCOPE_CONFLICT')
 bindings=verified['input_file_receipts']
 chunks=keyed(read_rows(bindings['chunks']),'chunk_id')
 golden=keyed(read_rows(bindings['golden']),'candidate_id')
 hybrid_all=keyed(read_rows(bindings['hybrid']),'candidate_id')
 require(len(chunks)==3488 and len(golden)==3813 and set(golden)==set(hybrid_all),'FROZEN_POPULATION_CONFLICT')
 locator=collections.defaultdict(set);types=collections.defaultdict(set)
 for cid,row in chunks.items():
  require(isinstance(row.get('content'),str) and row['content'].strip(),'CANONICAL_CHUNK_CONTENT_REQUIRED')
  for m in row['member_locators']:
   k=canonical(m['locator']);locator[k].add(cid);types[k].add(row['source_type'])
 gold={}
 for q,row in golden.items():
  relevant=set()
  for e in row['evidence_locators']:
   k=canonical(e['source_locator'])
   require(bool(locator[k]) and bool(types[k]&set(row['source_types'])),'GOLD_MAPPING_CONFLICT:'+q)
   relevant.update(locator[k])
  require(bool(relevant),'EMPTY_GOLD:'+q);gold[q]=relevant
 h=hashlib.sha256()
 for q in sorted(gold):
  raw=canonical({'candidate_id':q,'relevant_chunk_ids':sorted(gold[q])});h.update(len(raw).to_bytes(8,'big'));h.update(raw)
 require(h.hexdigest()==verified['gold_mapping']['recomputed_sha256'],'LOGICAL_GOLD_CONFLICT')
 require(collections.Counter(r['split'] for r in hybrid_all.values())=={'development':2664,'test':1149},'SPLIT_CONFLICT')
 ids=sorted(q for q in golden if hybrid_all[q]['split']=='development')
 dev={'ids':ids,'chunks':chunks,'gold':{q:gold[q] for q in ids},
      'questions':{q:golden[q]['evaluation_content']['question'] for q in ids},
      'source_types':{q:sorted(golden[q]['source_types']) for q in ids},
      'hybrid':{q:rank_ids(hybrid_all[q],100) for q in ids},'incumbent':{}}
 retained=keyed(read_rows(PINS['C_rankings']),'candidate_id')
 require(set(retained)==set(ids),'NB22C_COHORT_CONFLICT')
 for q,row in retained.items():
  order=row['ranked_chunk_ids']
  require(row['split']=='development' and row['variant']==prior['retained_variant'],'NB22C_SELECTION_OR_SPLIT_CONFLICT')
  require(len(order) in (50,100) and len(order)==len(set(order)) and set(order).issubset(dev['hybrid'][q]),'NB22C_ORDER_CONFLICT')
  require(all(abs(v-row['metrics'][k])<=1e-12 for k,v in metrics(order,dev['gold'][q]).items()),'NB22C_PER_QUERY_REPLAY_CONFLICT')
  dev['incumbent'][q]=order
 actual=mean_metric(metrics(dev['incumbent'][q],dev['gold'][q]) for q in ids)
 require(all(abs(actual[k]-prior['after'][k])<=1e-12 for k in actual),'NB22C_AGGREGATE_REPLAY_CONFLICT')
 diagnosis=keyed(read_rows(PINS['C_diagnosis']),'candidate_id')
 require(set(diagnosis)==set(ids),'DIAGNOSIS_COHORT_CONFLICT')
 cmap=components(dev);parts=partition(cmap)
 for name,qs in parts.items():
  require(all(diagnosis[q]['internal_partition']==name and diagnosis[q]['source_component']==cmap[q] for q in qs),'INTERNAL_SPLIT_CHANGED')
 STATE['memory'].update(dev=dev,partitions=parts,component_map=cmap,prior=prior,chunk_binding=bindings['chunks'])
 return {'subtasks':{'1.1':'EXACT_INPUT_HASHES_VERIFIED','1.2':'GOLD_AND_PRIOR_RANKINGS_REPLAYED','1.3':'SAME_INTERNAL_SOURCE_PARTITIONS'},
  'corpus_chunks':len(chunks),'development_queries':len(ids),'baseline':actual,
  'test_rows_read_for_identity_only':True,'historical_test_evaluation':False}
A01=run_action(1,action01)


{
  "action": 1,
  "subtasks": {
    "1.1": "EXACT_INPUT_HASHES_VERIFIED",
    "1.2": "GOLD_AND_PRIOR_RANKINGS_REPLAYED",
    "1.3": "SAME_INTERNAL_SOURCE_PARTITIONS"
  },
  "corpus_chunks": 3488,
  "development_queries": 2664,
  "baseline": {
    "hit_at_10": 0.8584834834834835,
    "hit_at_5": 0.8066816816816816,
    "ndcg_at_10": 0.6087143030504241,
    "ndcg_at_5": 0.5846432740221266,
    "precision_at_10": 0.1170795795795796,
    "precision_at_5": 0.2063813813813814,
    "recall_at_10": 0.6779356737690071,
    "recall_at_5": 0.6121113673197007,
    "reciprocal_rank": 0.699319256255785
  },
  "test_rows_read_for_identity_only": true,
  "historical_test_evaluation": false,
  "status": "COMPLETE_IN_SCOPE",
  "new_api_calls": 0
}


<div dir="rtl" align="right">

### NB22D-C005 — اقدام۲ — فریز روش جدید

۲٫۱ جست‌وجوی کل پیکره؛ ۲٫۲ قاعدهٔ انتخاب پوشش/رتبه؛ ۲٫۳ حفظ حدهای پذیرش. پایان: ثبت طرح پیش از نتیجه.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [5]:
# NB22D-C005 — Action 2: freeze a corpus-wide retrieval change.
def action02():
 plan={'schema_version':CONFIG['version'],'producer_code_sha256':NOTEBOOK_CODE_HASH,
  'source_inputs':STATE['inputs'],'configuration':CONFIG,'partitions':STATE['memory']['partitions'],
  'hypothesis':'Retrieve evidence beyond Hybrid100 using full canonical-corpus word BM25 and normalized within-word character trigram BM25. Preserve existing Hybrid100 in every expanded pool.',
  'analyzer':'NFKC; Arabic yeh/kaf to Persian; Persian/Arabic digits to ASCII; remove combining marks/tatweel; ZWNJ to space; Unicode word tokens. Word BM25 has no stopword/stemming; character trigrams are within words, unique per query, skip df>30% corpus.',
  'pool_rule':'Keep all original Hybrid100. Fill to requested upper bound using k60 reciprocal rank fusion of selected new channels. Never fill with zero-score documents or Gold-derived choices.',
  'coverage_selection':'Tuning only: highest macro min(10,gold-in-pool)/gold-count; tie smaller mean actual depth then name. This is a pool ceiling, not final Recall@10.',
  'ranking_selection':'Tuning only: highest nDCG@10 among strict nDCG gains with Recall@10 and Hit@10 nondecreasing against NB22C. One frozen winner or incumbent.',
  'ranking_rule':'RRF k10, incumbent weight .75, remaining .25 split equally over lexical channels; absent-channel terms contribute zero. Lexical lists restricted to positive-score top800.',
  'validation':'Evaluate only frozen coverage winner and ranking winner on same previously exposed Development validation; no runner-up selection. Improvement exploratory, no independent final acceptance.',
  'readiness_for_new_reranker':'Coverage ceiling on internal validation >=.85 and not below seed-pool ceiling; prerequisite only, not a quality PASS.',
  'metrics_acceptance':{'recall_at_10':'>=.85','ndcg_at_10':'>=.75','hit_at_10':'NO_ABSOLUTE_CONTRACT_THRESHOLD; comparison-only no-regression'},
  'model_execution':False,'new_api_calls':0,'new_model_pairs':0,'historical_test_evaluation':False,
  'cache':'Versioned hash-bound per-query lexical ranking, atomic/reload; reuse exact cached ranks; cache miss is CPU lexical search only, never model/network.',
  'health_deferral_preserved':True,'release_acceptance_granted':False}
 receipt=save('A02.frozen_plan',plan);STATE['memory'].update(plan=plan,plan_receipt=receipt)
 return {'subtasks':{'2.1':'FROZEN_NEW_RETRIEVAL_METHOD','2.2':'FROZEN_COVERAGE_AND_RANKING_RULES','2.3':'PRESERVED_ACCEPTANCE_THRESHOLDS'},'plan':receipt,
  'pool_depth_upper_bounds':[400,800],'whole_corpus':True,'new_api_calls':0}
A02=run_action(2,action02,(1,))


{
  "action": 2,
  "subtasks": {
    "2.1": "FROZEN_NEW_RETRIEVAL_METHOD",
    "2.2": "FROZEN_COVERAGE_AND_RANKING_RULES",
    "2.3": "PRESERVED_ACCEPTANCE_THRESHOLDS"
  },
  "plan": {
    "path": "reports/phase2/quality_improvement/NB22D.v1/A02.frozen_plan.3dca62c066918f280dd750c3792ea04fe06fd4382acbcaf020d4a034f4444985.json",
    "file_bytes_sha256": "3dca62c066918f280dd750c3792ea04fe06fd4382acbcaf020d4a034f4444985",
    "bytes": 130850,
    "reload_validation": "PASS"
  },
  "pool_depth_upper_bounds": [
    400,
    800
  ],
  "whole_corpus": true,
  "new_api_calls": 0,
  "status": "COMPLETE_IN_SCOPE"
}


<div dir="rtl" align="right">

### NB22D-C006 — اقدام۳ — اجرای بازیابی

۳٫۱ جست‌وجوی پیکره؛ ۳٫۲ کش/Resume هر سؤال؛ ۳٫۳ کنترل شناسهٔ تمام کانال‌ها. پایان: دو فهرست برای تمام Development؛ مدل تازه صفر.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [6]:
# NB22D-C006 — Action 3: full corpus lexical retrieval, checkpointed per query.
def normalized_words(text):
 text=unicodedata.normalize('NFKC',text).lower().translate(str.maketrans('يك٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹','یک01234567890123456789'))
 text=''.join(c for c in text if not unicodedata.combining(c) and c!='ـ').replace('\u200c',' ')
 return re.findall(r'[^\W_]+',text,flags=re.UNICODE)

def char_tokens(words):
 return [w[i:i+3] for w in words if len(w)>=3 for i in range(len(w)-2)]

def build_index(chunks,mode):
 ids=sorted(chunks);postings=collections.defaultdict(list);lengths=[]
 for i,cid in enumerate(ids):
  words=normalized_words(chunks[cid]['content']);tokens=words if mode=='word' else char_tokens(words)
  counts=collections.Counter(tokens);lengths.append(sum(counts.values()))
  for token,tf in counts.items():postings[token].append((i,tf))
 require(sum(lengths)>0,'EMPTY_LEXICAL_INDEX:'+mode)
 if mode=='char':postings={t:p for t,p in postings.items() if len(p)/len(ids)<=CONFIG['char_max_df_fraction']}
 return {'ids':ids,'postings':postings,'lengths':lengths,'avgdl':sum(lengths)/len(ids)}

def lexical_search(index,question,mode):
 words=normalized_words(question)
 tokens=words if mode=='word' else sorted(set(char_tokens(words)))
 scores={};n=len(index['ids']);k1=CONFIG['bm25']['k1'];b=CONFIG['bm25']['b']
 for token,qtf in collections.Counter(tokens).items():
  posting=index['postings'].get(token,());df=len(posting)
  if not df:continue
  idf=math.log(1+(n-df+.5)/(df+.5))
  for i,tf in posting:
   norm=1-b+b*index['lengths'][i]/index['avgdl']
   scores[i]=scores.get(i,0.0)+qtf*idf*tf*(k1+1)/(tf+k1*norm)
 return [index['ids'][i] for i in sorted(scores,key=lambda i:(-scores[i],index['ids'][i]))[:800] if scores[i]>0]

def expanded_pool(seed,channels,spec):
 score=collections.defaultdict(float)
 for name in spec['channels']:
  for i,c in enumerate(channels[name]):score[c]+=1/(60+i+1)
 tail=sorted(score,key=lambda c:(-score[c],c))
 out=list(seed);seen=set(seed)
 for c in tail:
  if len(out)>=spec['depth']:break
  if c not in seen:out.append(c);seen.add(c)
 require(len(out)==len(set(out)) and set(seed).issubset(out),'EXPANSION_DROPPED_SEED_OR_DUPLICATED')
 return out

def lexical_fusion(incumbent,channels,spec):
 score=collections.defaultdict(float);k=CONFIG['rrf_k'];w=CONFIG['incumbent_rank_weight']
 for i,c in enumerate(incumbent):score[c]+=w/(k+i+1)
 for name in spec['channels']:
  for i,c in enumerate(channels[name]):score[c]+=(1-w)/len(spec['channels'])/(k+i+1)
 return sorted(score,key=lambda c:(-score[c],c))

def action03():
 dev=STATE['memory']['dev'];start=time.monotonic();last=start;channels_by_query={};indexes={};reused=0;fresh=0
 # Bind cache to code, full corpus, questions and analyzer configuration, not Gold or outcomes.
 identity={'version':CONFIG['version'],'code':NOTEBOOK_CODE_HASH,'configuration':CONFIG,
  'corpus_sha256':binding_hash(STATE['memory']['chunk_binding'])}
 folder=path_for('artifacts/phase2/quality_improvement/NB22D.lexical_cache_v1');folder.mkdir(parents=True,exist_ok=True)
 for q in dev['ids']:
  require(time.monotonic()-start<CONFIG['maximum_local_seconds'],'LOCAL_TIME_CAP_REACHED:rerun_reuses_saved_queries')
  key={'identity':identity,'candidate_id':q,'question_sha256':digest(dev['questions'][q].encode())}
  path=folder/(digest(canonical(key))+'.json')
  if path.is_file():
   record=decode(path.read_bytes())
   require(record.get('key')==key and record.get('payload_sha256')==digest(canonical(record['channels'])),'LEXICAL_CACHE_IDENTITY_CONFLICT')
   ranks=record['channels'];reused+=1
  else:
   if not indexes:
    print(json.dumps({'stage':'BUILD_FULL_CORPUS_LEXICAL_INDEXES','chunks':len(dev['chunks']),'model_pairs':0}),flush=True)
    indexes={m:build_index(dev['chunks'],m) for m in ('word','char')}
   ranks={m:lexical_search(indexes[m],dev['questions'][q],m) for m in ('word','char')}
   record={'key':key,'channels':ranks,'payload_sha256':digest(canonical(ranks))};raw=canonical(record)
   fd,name=tempfile.mkstemp(dir=folder,prefix='.nb22d-query-')
   try:
    with os.fdopen(fd,'wb') as f:f.write(raw);f.flush();os.fsync(f.fileno())
    try:os.link(name,path)
    except FileExistsError:require(path.read_bytes()==raw,'CONCURRENT_LEXICAL_CACHE_CONFLICT')
   finally:Path(name).unlink(missing_ok=True)
   require(decode(path.read_bytes())==record,'LEXICAL_CACHE_RELOAD_FAILURE');fresh+=1
  require(set(ranks)=={'word','char'},'LEXICAL_CACHE_CHANNEL_CONFLICT')
  for order in ranks.values():
   require(len(order)<=800 and len(order)==len(set(order)) and set(order).issubset(dev['chunks']),'LEXICAL_CACHE_POOL_CONFLICT')
  channels_by_query[q]=ranks
  if time.monotonic()-last>=20 or len(channels_by_query)==len(dev['ids']):
   print(json.dumps({'stage':'FULL_CORPUS_RETRIEVAL','completed':len(channels_by_query),'total':len(dev['ids']),
    'fresh_cpu_queries':fresh,'reused_queries':reused,'elapsed_seconds':round(time.monotonic()-start,1),'new_api_calls':0}),flush=True);last=time.monotonic()
 receipt=save('A03.lexical_channels',[{'candidate_id':q,'channels':channels_by_query[q]} for q in dev['ids']],rows=True)
 STATE['memory']['channels']=channels_by_query
 return {'subtasks':{'3.1':'SEARCHED_COMPLETE_CORPUS','3.2':'SAVED_AND_REUSED_PER_QUERY_CACHE','3.3':'VERIFIED_ALL_CHANNEL_IDS'},
  'queries':len(channels_by_query),'fresh_cpu_queries':fresh,'reused_queries':reused,'artifact':receipt,'new_model_pairs':0}
A03=run_action(3,action03,(1,2))


{"stage": "BUILD_FULL_CORPUS_LEXICAL_INDEXES", "chunks": 3488, "model_pairs": 0}
{"stage": "FULL_CORPUS_RETRIEVAL", "completed": 918, "total": 2664, "fresh_cpu_queries": 918, "reused_queries": 0, "elapsed_seconds": 20.0, "new_api_calls": 0}
{"stage": "FULL_CORPUS_RETRIEVAL", "completed": 1963, "total": 2664, "fresh_cpu_queries": 1963, "reused_queries": 0, "elapsed_seconds": 40.0, "new_api_calls": 0}
{"stage": "FULL_CORPUS_RETRIEVAL", "completed": 2664, "total": 2664, "fresh_cpu_queries": 2664, "reused_queries": 0, "elapsed_seconds": 53.4, "new_api_calls": 0}
{
  "action": 3,
  "subtasks": {
    "3.1": "SEARCHED_COMPLETE_CORPUS",
    "3.2": "SAVED_AND_REUSED_PER_QUERY_CACHE",
    "3.3": "VERIFIED_ALL_CHANNEL_IDS"
  },
  "queries": 2664,
  "fresh_cpu_queries": 2664,
  "reused_queries": 0,
  "artifact": {
    "path": "reports/phase2/quality_improvement/NB22D.v1/A03.lexical_channels.41d1e2481910dee526c066ee807bc27ce22486d841b150bacace4c8c8ab0aa82.jsonl",
    "file_bytes_sha256": "41d1e2481

<div dir="rtl" align="right">

### NB22D-C007 — اقدام۴ — انتخاب روی tuning

۴٫۱ مقایسهٔ پوشش Poolها؛ ۴٫۲ مقایسهٔ رتبه‌بندی؛ ۴٫۳ فریز دو برندهٔ مستقل هر شاخه. پایان: انتخاب بدون نتیجهٔ validation.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [7]:
# NB22D-C007 — Action 4: choose coverage and ranking candidates on tuning only.
def coverage(pool,gold):
 return min(10,len(set(pool)&gold))/len(gold)

def action04():
 dev=STATE['memory']['dev'];tuning=STATE['memory']['partitions']['tuning'];channels=STATE['memory']['channels']
 pool_rows=[];pools={};ranks={};rank_rows=[];rank_metrics={}
 for spec in CONFIG['pool_specs']:
  orders={q:expanded_pool(dev['hybrid'][q],channels[q],spec) for q in dev['ids']};pools[spec['name']]=orders
  values=[coverage(orders[q],dev['gold'][q]) for q in tuning]
  pool_rows.append({'variant':spec['name'],'tuning_pool_ceiling':statistics.fmean(values),
   'mean_actual_depth':statistics.fmean(len(orders[q]) for q in tuning),'queries':len(tuning)})
 base=mean_metric(metrics(dev['incumbent'][q],dev['gold'][q]) for q in tuning)
 for spec in CONFIG['ranking_specs']:
  orders={q:lexical_fusion(dev['incumbent'][q],channels[q],spec) for q in dev['ids']};ranks[spec['name']]=orders
  values={q:metrics(orders[q],dev['gold'][q]) for q in dev['ids']};rank_metrics[spec['name']]=values
  actual=mean_metric(values[q] for q in tuning)
  rank_rows.append({'variant':spec['name'],'metrics':actual,'delta':{k:actual[k]-base[k] for k in base},'eligible':eligible(actual,base)})
 pool_winner=sorted(pool_rows,key=lambda r:(-r['tuning_pool_ceiling'],r['mean_actual_depth'],r['variant']))[0]['variant']
 candidates=[r for r in rank_rows if r['eligible']]
 rank_winner=sorted(candidates,key=lambda r:(-r['metrics']['ndcg_at_10'],r['variant']))[0]['variant'] if candidates else None
 freeze={'plan':STATE['memory']['plan_receipt'],'pool_winner':pool_winner,'ranking_winner':rank_winner,
  'coverage_comparisons':pool_rows,'ranking_comparisons':rank_rows,'reference':base,'selection_scope':'TUNING_ONLY',
  'validation_used_for_selection':False}
 receipt=save('A04.selection_freeze',freeze)
 STATE['memory'].update(pools=pools,ranks=ranks,rank_metrics=rank_metrics,pool_winner=pool_winner,rank_winner=rank_winner,selection_receipt=receipt)
 return {'subtasks':{'4.1':'COMPARED_EXPANDED_POOLS','4.2':'COMPARED_LEXICAL_RANKINGS','4.3':'FROZEN_WINNERS_BEFORE_VALIDATION'},**freeze,'freeze':receipt}
A04=run_action(4,action04,(1,2,3))


{
  "action": 4,
  "subtasks": {
    "4.1": "COMPARED_EXPANDED_POOLS",
    "4.2": "COMPARED_LEXICAL_RANKINGS",
    "4.3": "FROZEN_WINNERS_BEFORE_VALIDATION"
  },
  "plan": {
    "path": "reports/phase2/quality_improvement/NB22D.v1/A02.frozen_plan.3dca62c066918f280dd750c3792ea04fe06fd4382acbcaf020d4a034f4444985.json",
    "file_bytes_sha256": "3dca62c066918f280dd750c3792ea04fe06fd4382acbcaf020d4a034f4444985",
    "bytes": 130850,
    "reload_validation": "PASS"
  },
  "pool_winner": "dual800",
  "ranking_winner": "char_rrf",
  "coverage_comparisons": [
    {
      "variant": "word400",
      "tuning_pool_ceiling": 0.8703349110728629,
      "mean_actual_depth": 397.6736947791165,
      "queries": 1992
    },
    {
      "variant": "char400",
      "tuning_pool_ceiling": 0.8752205249569708,
      "mean_actual_depth": 398.3092369477912,
      "queries": 1992
    },
    {
      "variant": "dual400",
      "tuning_pool_ceiling": 0.8807736899980876,
      "mean_actual_depth": 399.592369477911

<div dir="rtl" align="right">

### NB22D-C008 — اقدام۵ — بررسی و گزارش

۵٫۱ بررسی برنده‌های فریز‌شده؛ ۵٫۲ جدول حدهای پذیرش و خطاهای پوشش؛ ۵٫۳ خروجی Pool/رتبه و گام بعدی. پایان: تصمیم روشن؛ نتیجهٔ منفی و موانع حفظ می‌شوند.

بستهٔ کامل همین مرحله؛ مراحل وابسته با ورودی نامعتبر مسدود می‌شوند.

</div>

In [8]:
# NB22D-C008 — Action 5: internal validation and visible acceptance tables.
def paired_component_intervals(ids, candidate, reference, component_map):
    import random
    names = ['recall_at_10', 'ndcg_at_10', 'hit_at_10']
    groups = collections.defaultdict(list)
    for q in ids:
        groups[component_map[q]].append(q)
    ordered = [groups[c] for c in sorted(groups)]
    sizes = [len(g) for g in ordered]
    sums = [[math.fsum(candidate[q][k] - reference[q][k] for q in g) for k in names] for g in ordered]
    rng = random.Random(CONFIG['seed']); samples = {k: [] for k in names}; repetitions = 2000
    for _ in range(repetitions):
        chosen = [rng.randrange(len(ordered)) for _ in ordered]
        denominator = sum(sizes[i] for i in chosen)
        for j, name in enumerate(names):
            samples[name].append(math.fsum(sums[i][j] for i in chosen) / denominator)
    def quantile(values, p):
        values = sorted(values); position = (len(values) - 1) * p
        lo = int(position); hi = min(lo + 1, len(values) - 1)
        return values[lo] + (values[hi] - values[lo]) * (position - lo)
    return {'method': 'PAIRED_SOURCE_COMPONENT_BOOTSTRAP', 'repetitions': repetitions,
        'component_count': len(ordered), 'confidence': 0.95,
        'scope': 'INTERNAL_EXPLORATORY_PREVIOUSLY_EXPOSED_DEVELOPMENT',
        'familywise_significance_claimed': False,
        'intervals': {k: {'lower': quantile(v, .025), 'upper': quantile(v, .975)} for k, v in samples.items()}}

def acceptance_table(before,after):
 rows=[]
 for key,title in [('recall_at_10','Recall@10'),('ndcg_at_10','nDCG@10'),('hit_at_10','Hit@10')]:
  threshold=CONFIG['thresholds'].get(key)
  rows.append({'metric':title,'before':before[key],'after':after[key],'change':after[key]-before[key],
   'acceptance_threshold':threshold,'absolute_status':('PASS' if after[key]>=threshold else 'FAIL') if threshold is not None else 'NO_ABSOLUTE_THRESHOLD',
   'acceptance_mark':'✅' if threshold is not None and after[key]>=threshold else '',
   'scope':'DEVELOPMENT_INTERNAL_NOT_FINAL_RELEASE_ACCEPTANCE',
   'no_regression_vs_previous':after[key]>=before[key]-1e-12})
 return rows

def print_table(rows):
 print('\n| معیار | قبل | بعد | حد پذیرش | قبول |')
 print('|---|---:|---:|---:|:---:|')
 for r in rows:
  threshold=f"{100*r['acceptance_threshold']:.2f}%" if r['acceptance_threshold'] is not None else 'در قرارداد تعیین نشده'
  print(f"| {r['metric']} | {100*r['before']:.2f}% | {100*r['after']:.2f}% | {threshold} | {r['acceptance_mark']} |")

def action05():
 dev=STATE['memory']['dev'];validation=STATE['memory']['partitions']['validation'];pw=STATE['memory']['pool_winner'];rw=STATE['memory']['rank_winner']
 selected_pool=STATE['memory']['pools'][pw]
 seed_ceiling=statistics.fmean(coverage(dev['hybrid'][q],dev['gold'][q]) for q in validation)
 expanded_ceiling=statistics.fmean(coverage(selected_pool[q],dev['gold'][q]) for q in validation)
 readiness=expanded_ceiling>=CONFIG['thresholds']['recall_at_10'] and expanded_ceiling>=seed_ceiling-1e-12
 base={q:metrics(dev['incumbent'][q],dev['gold'][q]) for q in dev['ids']}
 values=STATE['memory']['rank_metrics'][rw] if rw else base
 before=mean_metric(base[q] for q in validation);candidate=mean_metric(values[q] for q in validation)
 passes=rw is not None and eligible(candidate,before)
 retained=rw if passes else STATE['memory']['prior']['retained_variant']
 kept=values if passes else base
 retained_orders=STATE['memory']['ranks'][rw] if passes else dev['incumbent']
 errors=[]
 for q in validation:
  old=set(dev['hybrid'][q]);new=set(selected_pool[q]);g=dev['gold'][q]
  if (g&new)-(g&old) or not (g&new):
   errors.append({'candidate_id':q,'source_types':dev['source_types'][q],
    'gold_count':len(g),'seed_gold_count':len(g&old),'expanded_gold_count':len(g&new),
    'recovered_gold_chunk_ids':sorted((g&new)-(g&old)),
    'still_missing_gold_chunk_ids':sorted(g-new)})
 slices={}
 for label in sorted({','.join(dev['source_types'][q]) for q in validation}):
  qs=[q for q in validation if ','.join(dev['source_types'][q])==label]
  slices[label]={'queries':len(qs),'seed_pool_ceiling':statistics.fmean(coverage(dev['hybrid'][q],dev['gold'][q]) for q in qs),
   'expanded_pool_ceiling':statistics.fmean(coverage(selected_pool[q],dev['gold'][q]) for q in qs),
   'ranking_before':mean_metric(base[q] for q in qs),'ranking_candidate':mean_metric(values[q] for q in qs)}
 pool_receipt=save('A05.selected_expanded_development_pool',[{'candidate_id':q,'split':'development','variant':pw,'candidate_chunk_ids':selected_pool[q]} for q in dev['ids']],rows=True)
 rank_receipt=save('A05.retained_development_rankings',[{'candidate_id':q,'split':'development','variant':retained,'ranked_chunk_ids':retained_orders[q],'metrics':kept[q]} for q in dev['ids']],rows=True)
 error_receipt=save('A05.coverage_recoveries_and_misses',errors,rows=True)
 full_before=mean_metric(base.values());full_after=mean_metric(kept.values())
 next_step=('BOUNDED_LOCAL_RERANKER_PILOT_ON_NEW_POOL_NO_AUTOMATIC_EXECUTION' if readiness else
  'POOL_CEILING_STILL_BELOW_TARGET_REVIEW_MISSES_AND_EXISTING_DENSE_CACHE_BEFORE_MORE_INFERENCE')
 rows=acceptance_table(full_before,full_after)
 summary={'schema_version':'NB22D_FINAL_V1','status':'CORPUS_EXPANSION_EXPERIMENT_COMPLETE_ACCEPTANCE_NOT_ESTABLISHED',
  'selected_pool':pw,'frozen_ranking_candidate':rw,'retained_ranking':retained,'ranking_passes_internal_guard':passes,
  'validation_pool_seed_ceiling':seed_ceiling,'validation_expanded_pool_ceiling':expanded_ceiling,
  'validation_pool_target':.85,'pool_ready_for_bounded_reranker_pilot':readiness,
  'pool_ceiling_is_not_actual_recall_at_10':True,
  'validation_ranking_before':before,'validation_ranking_candidate':candidate,
  'internal_uncertainty':paired_component_intervals(validation,values,base,STATE['memory']['component_map']) if rw else None,
  'source_slices':slices,'full_development_before':full_before,'full_development_retained':full_after,
  'acceptance_table':rows,'next_step':next_step,
  'coverage_recoveries':{'validation_queries_with_new_gold':sum(bool(r['recovered_gold_chunk_ids']) for r in errors),
   'validation_queries_still_without_any_gold':sum(r['expanded_gold_count']==0 for r in errors)},
  'pool_artifact':pool_receipt,'retained_rankings':rank_receipt,'coverage_cases':error_receipt,
  'source_inputs':STATE['inputs'],'producer_code_sha256':NOTEBOOK_CODE_HASH,
  'plan':STATE['memory']['plan_receipt'],'selection':STATE['memory']['selection_receipt'],
  'scope':'PREVIOUSLY_EXPOSED_DEVELOPMENT_INTERNAL_EVALUATION_NOT_INDEPENDENT_FINAL_HOLDOUT',
  'unchanged_open_work':['independent final evaluation','context/generation semantic and safety qualification','NB21 action48 and health deferral','prior held API responses and unknown HTTP504 remain untouched'],
  'historical_test_evaluation':False,'release_acceptance_granted':False,'new_api_calls':0,'new_model_pairs':0,
  'historical_inputs_modified':False,'artifacts':list(STATE['artifacts'])}
 for r in summary['artifacts']:
  require(digest(path_for(r['path']).read_bytes())==r['file_bytes_sha256'],'FINAL_ARTIFACT_INTEGRITY_CONFLICT')
 receipt=save('A05.final_report',summary)
 STATE['memory']['final']=summary
 print_table(rows)
 return {'subtasks':{'5.1':'VALIDATED_FROZEN_CANDIDATES','5.2':'REPORTED_REAL_THRESHOLDS_AND_COVERAGE','5.3':'EXPORTED_NEXT_PILOT_INPUTS_AND_PRESERVED_GATES'},
  'retained_ranking':retained,'ranking_passes_internal_guard':passes,
  'validation_pool_seed_ceiling':seed_ceiling,'validation_expanded_pool_ceiling':expanded_ceiling,
  'pool_ready_for_bounded_reranker_pilot':readiness,'acceptance_table':rows,
  'next_step':next_step,'report':receipt,'release_acceptance_granted':False}
A05=run_action(5,action05,(1,2,3,4))
print('\n========== NB22D — نتیجهٔ نهایی ==========')
if A05['status']=='COMPLETE_IN_SCOPE':
 print_table(A05['acceptance_table'])
 print(json.dumps(A05,ensure_ascii=False,indent=2))
else:
 print(json.dumps({'status':'INCOMPLETE_NO_ACCEPTANCE','actions':{str(k):v for k,v in STATE['actions'].items()},
  'instruction_fa':'فقط فایل یا قرارداد مشخص‌شده را بررسی کن؛ هیچ مدل یا API را دوباره اجرا نکن.',
  'release_acceptance_granted':False,'new_api_calls':0},ensure_ascii=False,indent=2))



| معیار | قبل | بعد | حد پذیرش | قبول |
|---|---:|---:|---:|:---:|
| Recall@10 | 67.79% | 68.35% | 85.00% |  |
| nDCG@10 | 60.87% | 61.16% | 75.00% |  |
| Hit@10 | 85.85% | 86.30% | در قرارداد تعیین نشده |  |
{
  "action": 5,
  "subtasks": {
    "5.1": "VALIDATED_FROZEN_CANDIDATES",
    "5.2": "REPORTED_REAL_THRESHOLDS_AND_COVERAGE",
    "5.3": "EXPORTED_NEXT_PILOT_INPUTS_AND_PRESERVED_GATES"
  },
  "retained_ranking": "char_rrf",
  "ranking_passes_internal_guard": true,
  "validation_pool_seed_ceiling": 0.8278769841269842,
  "validation_expanded_pool_ceiling": 0.9429563492063492,
  "pool_ready_for_bounded_reranker_pilot": true,
  "acceptance_table": [
    {
      "metric": "Recall@10",
      "before": 0.6779356737690071,
      "after": 0.6835327887411221,
      "change": 0.005597114972115014,
      "acceptance_threshold": 0.85,
      "absolute_status": "FAIL",
      "acceptance_mark": "",
      "scope": "DEVELOPMENT_INTERNAL_NOT_FINAL_RELEASE_ACCEPTANCE",
      "no_regression_vs_prev

<div dir="rtl" align="right">

## اعتبارسنجی تحویل

۲۳ کنترل نرم‌افزاری با دادهٔ مصنوعی و شبکهٔ مسدود پاس شده است: اجرای ترتیبی هشت سل در فضای تازه، بازیابی شاهد بیرون Pool100، حفظ seed، سنجه و normalization، تکرار با کش کامل، فایل ناقص و کش خراب، علامت پذیرش و Schema نوت‌بوک. این شاهد، اجرای پروژه روی دستگاه کاربر یا تضمین بهبود کیفیت نیست.

فایل‌های واقعی C056، Chunk و نتایج NB22C روی دستگاه کاربر مصرف می‌شوند؛ نتیجهٔ اصلی پس از اجرای همان ورودی‌ها معلوم می‌شود. کش رتبه‌های واژه/نویسه و Pool بزرگ‌تر می‌توانند چندصد مگابایت فضا بگیرند؛ این‌ها خروجی‌های نسخه‌دار جدیدند و Cache قدیمی را حذف یا بازنویسی نمی‌کنند.

</div>